In [9]:
import numpy as np

# Dados dos 15 Microsserviços (Valor de Negócio, RAM em GB, CPU em Cores)
# Gerados para testar o cenário
servicos = np.array([
    [100, 2.0, 1], [150, 4.0, 2], [90,  1.5, 1], [200, 5.0, 3],
    [80,  1.0, 1], [120, 3.0, 2], [110, 2.0, 1], [170, 4.5, 2],
    [60,  1.0, 1], [130, 2.5, 2], [140, 3.5, 1], [190, 5.5, 3],
    [75,  1.5, 1], [105, 2.0, 1], [160, 4.0, 2]
])

MAX_RAM = 16.0
MAX_CPU = 8

def avaliar_fitness(individuo, estrategia):
    """Calcula o fitness com base na estratégia escolhida (A ou B)."""
    selecionados = servicos[individuo == 1]

    if len(selecionados) == 0:
        return 0

    valor_total = np.sum(selecionados[:, 0])
    ram_total = np.sum(selecionados[:, 1])
    cpu_total = np.sum(selecionados[:, 2])

    excesso_ram = max(0, ram_total - MAX_RAM)
    excesso_cpu = max(0, cpu_total - MAX_CPU)

    # Estratégia A: Penalidade Rígida (fitness = 0 se violar limites)
    if estrategia == 'A':
        if excesso_ram > 0 or excesso_cpu > 0:
            return 0
        return valor_total

    # Estratégia B: Penalidade Proporcional (redução baseada no excesso)
    elif estrategia == 'B':
        penalidade = (excesso_ram * 20) + (excesso_cpu * 30)
        fitness_final = valor_total - penalidade
        return max(0, fitness_final) # Evita fitness negativo

def torneio(populacao, fitness, k=3):
    idx = np.random.choice(len(populacao), k, replace=False)
    melhor_idx = idx[np.argmax(fitness[idx])]
    return populacao[melhor_idx]

def crossover_ponto_unico(pai1, pai2):
    ponto = np.random.randint(1, len(pai1))
    filho1 = np.concatenate([pai1[:ponto], pai2[ponto:]])
    filho2 = np.concatenate([pai2[:ponto], pai1[ponto:]])
    return filho1, filho2

def mutacao_binaria(individuo, taxa=0.05):
    para_mutar = np.random.rand(len(individuo)) < taxa
    individuo[para_mutar] = 1 - individuo[para_mutar]
    return individuo

def executar_ag(estrategia, pop_size=50, geracoes=100):
    populacao = np.random.randint(2, size=(pop_size, 15))
    historico_media = []
    historico_std = []

    melhor_global = None
    melhor_fitness_global = -1

    for _ in range(geracoes):
        fitness = np.array([avaliar_fitness(ind, estrategia) for ind in populacao])

        historico_media.append(np.mean(fitness))
        historico_std.append(np.std(fitness))

        melhor_idx = np.argmax(fitness)
        if fitness[melhor_idx] > melhor_fitness_global:
            melhor_fitness_global = fitness[melhor_idx]
            melhor_global = populacao[melhor_idx]

        nova_populacao = []
        while len(nova_populacao) < pop_size:
            pai1 = torneio(populacao, fitness)
            pai2 = torneio(populacao, fitness)
            f1, f2 = crossover_ponto_unico(pai1, pai2)
            nova_populacao.extend([mutacao_binaria(f1), mutacao_binaria(f2)])

        populacao = np.array(nova_populacao[:pop_size])

    return melhor_global, melhor_fitness_global, historico_media, historico_std

# Execução e Testes
print("LAB 02 - ESTRATÉGIA A (Rígida)")
best_A, fit_A, media_A, std_A = executar_ag('A')
print(f"Melhor Combinação: {best_A}\nFitness: {fit_A}")
print(f"Média/Std finais: {media_A[-1]:.2f} / {std_A[-1]:.2f}\n")

print("LAB 02 - ESTRATÉGIA B (Proporcional)")
best_B, fit_B, media_B, std_B = executar_ag('B')
print(f"Melhor Combinação: {best_B}\nFitness: {fit_B}")
print(f"Média/Std finais: {media_B[-1]:.2f} / {std_B[-1]:.2f}")

LAB 02 - ESTRATÉGIA A (Rígida)
Melhor Combinação: [0 0 1 0 1 0 1 1 0 0 1 0 1 1 0]
Fitness: 770.0
Média/Std finais: 399.20 / 318.43

LAB 02 - ESTRATÉGIA B (Proporcional)
Melhor Combinação: [1 1 1 1 1 1 1 1 1 1 1 0 1 1 1]
Fitness: 870.0
Média/Std finais: 852.40 / 22.52
